# Speech Command Recognition Subsystem for Autonomous Warehouse Robots
**Developer:** Viktoriia Hlushkova

Цей блокнот містить фінальний код розробленої підсистеми.
Він включає:

1. Ініціалізацію середовища та бібліотек.

In [ ]:
# === БЛОК 1: ІНІЦІАЛІЗАЦІЯ ТА НАЛАШТУВАННЯ СЕРЕДОВИЩА ===
import os
import warnings
import time
import datetime
import torch
import transformers
import evaluate
import librosa
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration, WhisperTokenizer
from huggingface_hub import login
from gtts import gTTS

# Приховуємо зайвий системний шум (про формат аудіо та attention mask)
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

print("╭" + "─"*60 + "╮")
print("│ ⚙️ НАЛАШТУВАННЯ СЕРЕДОВИЩА...                          │")
print("╰" + "─"*60 + "╯")

# Тихе встановлення необхідних бібліотек
os.system("pip install -q transformers==4.44.2 datasets==2.16.0 accelerate==0.33.0 huggingface_hub")
os.system("pip install -q librosa evaluate jiwer soundfile gTTS SpeechRecognition")

# Конфігурація токена доступу Hugging Face для користувачів GitHub
HF_TOKEN = ""  # Вставте свій токен авторизації тут
try:
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("  ✅ Авторизація Hugging Face успішна!\n")
    else:
        print("  ⚠️ Токен Hugging Face не введено. Використовується базовий публічний доступ.\n")
except Exception as e:
    print(f"  ❌ Помилка авторизації: {e}\n")

# Визначення пристрою для обчислень (GPU/CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  ✅ Використовується пристрій: {device}\n")

2. Завантаження донавченої (Fine-Tuned) моделі OpenAI Whisper Small.


In [ ]:
# === БЛОК 2: ЗАВАНТАЖЕННЯ ДОНАВЧЕНОЇ МОДЕЛІ ===
MODEL_PATH = "./whisper-fine-tuned/final" # Шлях до твоєї збереженої моделі

print("✨ Завантаження твоєї донавченої моделі...")

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Ukrainian", task="transcribe")
forced_ids = processor.get_decoder_prompt_ids(language="uk", task="transcribe")

try:
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH).to(device)
    print("  ✅ Донавчену модель успішно завантажено!\n")
except Exception as e:
    print(f"  ❌ Помилка завантаження моделі. Перевір шлях: {MODEL_PATH}\n  Деталі: {e}")


3. Алгоритм логічного контролера безпеки (Fail-Safe).

In [ ]:
# === БЛОК 3: ЛОГІЧНИЙ КОНТРОЛЕР БЕЗПЕКИ ===
def robot_brain(text, mode="log"):
    """
    mode="log": повертає розгорнуту текстову відповідь для інтерфейсу.
    mode="intent": повертає лише клас наміру для кількісних тестів.
    """
    text = text.lower()
    timestamp = datetime.datetime.now().strftime("%H:%M:%S")

    # Базова логіка визначення наміру (як у твоїй версії 5)
    if "стій" in text or "стоп" in text or "зупинись" in text:
        intent = "CRITICAL"
    elif "вперед" in text or "рухайся" in text:
        intent = "MOVE"
    elif "ліво" in text or "право" in text:
        intent = "TURN"
    elif "живлення" in text or "батарея" in text or "маніпулятора" in text:
        intent = "STATUS"
    else:
        intent = "ERROR"

    # Для 5-го розділу: повертаємо лише сухий намір
    if mode == "intent":
        return intent

    # Для 4-го розділу: повертаємо красивий лог системи
    if intent == "CRITICAL":
        return f"[{timestamp}] 🔴 [CRITICAL]: Переривання руху! Екстрене гальмування."
    elif intent == "MOVE":
        return f"[{timestamp}] 🟢 [MOVE]: Активація маршових двигунів. Вектор: ВПЕРЕД."
    elif intent == "TURN":
         return f"[{timestamp}] 🔵 [TURN]: Зміна траєкторії."
    elif intent == "STATUS":
        return f"[{timestamp}] 🔋 [STATUS]: Рівень заряду акумулятора: 87%. Система в нормі."
    else:
        return f"[{timestamp}] ⚠️ [ERROR]: Команду не розпізнано. Фільтрація шуму."

4. Проведення експериментів на реальних зашумлених аудіозаписах (Inference).


In [ ]:
# === БЛОК 4: ЕКСПЕРИМЕНТИ НА РЕАЛЬНИХ ЗАПИСАХ ===
test_files = [
    "forward_clean.m4a",
    "service_clean.m4a",
    "stop_clean.m4a",
    "forward_light.m4a",
    "service_light.m4a",
    "stop_light.m4a",
    "forward_extreme.m4a",
    "service_extreme.m4a",
    "stop_extreme.m4a"
]

print("🚀 ЗАПУСК ЕКСПЕРИМЕНТУ НА РЕАЛЬНИХ ЗАПИСАХ ІЗ ШУМОМ...\n")

for file_name in test_files:
    print(f"╭{'─'*60}╮")
    print(f"│ 🎧 ФАЙЛ: {file_name:<49} │")
    print(f"╰{'─'*60}╯")

    try:
        # Завантажуємо запис
        speech, sr = librosa.load(file_name, sr=16000)

        # Розпізнаємо
        inputs = processor(speech, sampling_rate=16000, return_tensors="pt").input_features.to(device)
        with torch.no_grad():
            predicted_ids = model.generate(inputs, forced_decoder_ids=forced_ids)
            transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        print(f"  🤖 Whisper почув: '{transcription}'")
        print(f"  💻 Реакція робота: {robot_brain(transcription)}\n")

    except Exception as e:
        print(f"  ❌ Помилка з файлом {file_name}.\n  Деталі: {e}\n")

5. Генерацію вибірки для кількісного експерименту.

In [ ]:
# === БЛОК 5: КІЛЬКІСНИЙ ЕКСПЕРИМЕНТ (100 КОМАНД) ===
print("╭" + "─"*60 + "╮")
print("│ 🚀 СТАРТ КІЛЬКІСНОГО ЕКСПЕРИМЕНТУ (100 КОМАНД)         │")
print("╰" + "─"*60 + "╯")

# Завантажуємо метрику WER
wer_metric = evaluate.load("wer")

# Генерація вибірки (100 команд: 5 класів по 20 фраз)
phrases_dict = {
    "CRITICAL": ["терміново стоп", "стій негайно", "зупинись", "стоп", "робот стій"] * 4,
    "MOVE": ["робот рухайся вперед", "вперед", "рухайся", "їдь прямо", "прямо"] * 4,
    "TURN": ["поверни на ліво", "поверни на право", "ліворуч", "направо", "поворот"] * 4,
    "STATUS": ["перевір статус маніпулятора", "який рівень заряду батареї", "увімкни живлення", "статус", "батарея"] * 4,
    "ERROR": ["яка сьогодні погода", "зроби мені каву", "привіт як справи", "що ти робиш", "розкажи новини"] * 4
}

# Функція додавання білого шуму (SNR)
def add_noise(signal, snr_db):
    if snr_db is None:
        return signal # Чистий звук
    signal_power = np.mean(signal**2)
    noise_power = signal_power / (10**(snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), len(signal))
    return signal + noise

# Проведення експерименту
noise_levels = {"Clean": None, "Medium (10dB SNR)": 10, "Extreme (0dB SNR)": 0}
results = []

os.makedirs("test_audio", exist_ok=True)

# Проходимо по всіх рівнях шуму
for noise_name, snr in noise_levels.items():
    print(f"\n📊 Тестування умов: {noise_name}...")

    correct_intents = 0
    predictions = []
    references = []

    for true_intent, phrases in phrases_dict.items():
        for i, text in enumerate(phrases):
            # Синтезуємо аудіо
            file_path = f"test_audio/cmd_{true_intent}_{i}.mp3"
            if not os.path.exists(file_path):
                tts = gTTS(text, lang='uk')
                tts.save(file_path)

            # Завантажуємо та накладаємо шум
            speech, sr = librosa.load(file_path, sr=16000)
            noisy_speech = add_noise(speech, snr)

            # Розпізнаємо через Whisper
            inputs = processor(noisy_speech, sampling_rate=16000, return_tensors="pt").input_features.to(device)
            with torch.no_grad():
                predicted_ids = model.generate(inputs, forced_decoder_ids=forced_ids)
                transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()

            # Аналізуємо контролером (використовуємо режим "intent")
            predicted_intent = robot_brain(transcription, mode="intent")

            # Збираємо статистику
            if predicted_intent == true_intent:
                correct_intents += 1

            predictions.append(transcription)
            references.append(text)

    # Розрахунок метрик для даного рівня шуму
    current_wer = wer_metric.compute(predictions=predictions, references=references) * 100
    intent_accuracy = (correct_intents / 100) * 100

    results.append({
        "Умови шуму": noise_name,
        "WER (Помилки слів), %": round(current_wer, 2),
        "Точність контролера (Intent Accuracy), %": round(intent_accuracy, 2)
    })

# Вивід та збереження фінальної таблиці
df_results = pd.DataFrame(results)
print("\n" + "═"*60)
print("🏆 ФІНАЛЬНІ РЕЗУЛЬТАТИ ЕКСПЕРИМЕНТУ:")
print("═"*60)
print(df_results.to_string(index=False))

df_results.to_csv("experiment_100_commands.csv", index=False)
print("\n✅ Таблицю збережено у файл 'experiment_100_commands.csv'")